In [38]:
import requests
import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString
from shapely.ops import unary_union
from sklearn.preprocessing import MinMaxScaler
from pyproj import Transformer
import folium
import time
import io

# ─────────────────────────────────────────
# ZENTRALE KONFIGURATION - nur hier ändern
# ─────────────────────────────────────────

# Testgebiet: Frankfurt Nord (Nieder-Eschbach, Bonames, Harheim)
LON_MIN, LAT_MIN = 8.62, 50.17
LON_MAX, LAT_MAX = 8.75, 50.23

# Für Overpass API (lat,lon Format)
BBOX_OSM = f"{LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}"

# Für WFS (UTM Format)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:25832", always_xy=True)
X_MIN, Y_MIN = transformer.transform(LON_MIN, LAT_MIN)
X_MAX, Y_MAX = transformer.transform(LON_MAX, LAT_MAX)
BBOX_UTM = f"{X_MIN:.0f},{Y_MIN:.0f},{X_MAX:.0f},{Y_MAX:.0f}"

# Kartenmittelpunkt
KARTE_MITTE = [(LAT_MIN + LAT_MAX) / 2, (LON_MIN + LON_MAX) / 2]

# Geeignete Nutzungsarten für Solarparks
GEEIGNETE_NUTZUNG = ["Ackerland", "Grünland", "Streuobst"]

# API URLs
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
WFS_URL      = "https://geowebdienste.frankfurt.de/SGK_Flurstuecke"
HEADERS      = {"User-Agent": "solar-finder/1.0"}

print(f"Konfiguration geladen:")
print(f"  OSM BBOX:  {BBOX_OSM}")
print(f"  UTM BBOX:  {BBOX_UTM}")
print(f"  Kartenmitte: {KARTE_MITTE}")

Konfiguration geladen:
  OSM BBOX:  50.17,8.62,50.23,8.75
  UTM BBOX:  472863,5557601,482169,5564234
  Kartenmitte: [50.2, 8.684999999999999]


Testgebiet definiert:
  OSM BBOX:  50.17,8.62,50.23,8.75
  UTM BBOX:  472863,5557601,482169,5564234
  Kartenmitte: [50.2, 8.684999999999999]


In [39]:
def lade_osm(filter_tag, beschreibung, versuche=3):
    """Lädt OSM-Linien (Autobahnen, Bahn) und gibt GeoDataFrame zurück"""
    query = f"""
        [out:json];
        way[{filter_tag}]({BBOX_OSM});
        out geom;
    """
    for i in range(versuche):
        try:
            response = requests.get(
                OVERPASS_URL,
                params={"data": query},
                headers=HEADERS,
                timeout=30
            )
            if not response.text.strip():
                print(f"Versuch {i+1}: Leere Antwort, warte 5 Sekunden...")
                time.sleep(5)
                continue
            break
        except Exception as e:
            print(f"Versuch {i+1} fehlgeschlagen: {e}")
            time.sleep(5)

    data = response.json()
    rows = []
    for element in data['elements']:
        if 'geometry' not in element:
            continue
        coords = [(p['lon'], p['lat']) for p in element['geometry']]
        if len(coords) >= 2:
            rows.append({
                "id": element['id'],
                "typ": beschreibung,
                "geometry": LineString(coords)
            })

    gdf = gpd.GeoDataFrame(rows, crs="EPSG:4326")
    print(f"{beschreibung}: {len(gdf)} Abschnitte geladen")
    return gdf


def farbe_nach_score(score):
    """Gibt Farbe basierend auf Score zurück"""
    if score >= 0.66:
        return "green"
    elif score >= 0.33:
        return "orange"
    else:
        return "red"


def kategorie_nach_score(score):
    """Gibt Kategorie basierend auf Score zurück"""
    if score >= 0.66:
        return "Gut geeignet"
    elif score >= 0.33:
        return "Mittel geeignet"
    else:
        return "Wenig geeignet"

print("Funktionen geladen.")

Funktionen geladen.


In [40]:
# --- Autobahnen laden ---
autobahnen = lade_osm('"highway"="motorway"', "Autobahnen")

# --- Bahnlinien laden ---
bahnlinien = lade_osm('"railway"="rail"', "Bahnlinien")

# --- ALKIS Flurstücke laden ---
print("\nLade ALKIS Flurstücke vom WFS...")

response_wfs = requests.get(
    WFS_URL,
    params={
        "SERVICE": "WFS",
        "VERSION": "2.0.0",
        "REQUEST": "GetFeature",
        "TYPENAMES": "Amt62_Flurstuecke:Flurstueck",
        "BBOX": f"{BBOX_UTM},EPSG:25832",
        "SRSNAME": "EPSG:25832",
    },
    timeout=120
)

print(f"WFS Status: {response_wfs.status_code}")

# In GeoDataFrame laden
alle_flurstuecke = gpd.read_file(io.BytesIO(response_wfs.content))
alle_flurstuecke = alle_flurstuecke.set_crs("EPSG:25832", allow_override=True)
print(f"Flurstücke geladen: {len(alle_flurstuecke)}")

# Lokal filtern: Nutzungsart + Größe > 7 ha
alle_flurstuecke["area_ha"] = alle_flurstuecke.geometry.area / 10000

maske = (
    alle_flurstuecke["TNTXT"].str.contains(
        "|".join(GEEIGNETE_NUTZUNG),
        case=False, na=False
    )
    & (alle_flurstuecke["area_ha"] > 7)
)

flaechen = alle_flurstuecke[maske].copy()
print(f"Geeignete Flächen (>7ha, richtige Nutzung): {len(flaechen)}")

Autobahnen: 87 Abschnitte geladen
Bahnlinien: 146 Abschnitte geladen

Lade ALKIS Flurstücke vom WFS...
WFS Status: 200
Flurstücke geladen: 21241
Geeignete Flächen (>7ha, richtige Nutzung): 27


In [41]:
# Infrastruktur auf UTM umstellen
autobahnen_m = autobahnen.to_crs(epsg=25832)
bahnlinien_m = bahnlinien.to_crs(epsg=25832)

# Alle Linien zu je einer Geometrie zusammenfassen
autobahn_union = unary_union(autobahnen_m.geometry)
bahn_union     = unary_union(bahnlinien_m.geometry)

# Abstände berechnen
flaechen["dist_autobahn_m"] = flaechen.geometry.apply(
    lambda g: g.distance(autobahn_union)
)
flaechen["dist_bahn_m"] = flaechen.geometry.apply(
    lambda g: g.distance(bahn_union)
)

# Filter: Nähe zur Infrastruktur
flaechen = flaechen[
    (flaechen["dist_autobahn_m"] < 2000) |
    (flaechen["dist_bahn_m"] < 2000)
].copy()

print(f"Nach Infrastruktur-Filter (<2km): {len(flaechen)} Flächen")

# Scoring
flaechen["score_flaeche"] = flaechen["area_ha"]
flaechen["score_infra"]   = 1 / (flaechen["dist_autobahn_m"] + 1)

scaler = MinMaxScaler()
flaechen[["score_flaeche", "score_infra"]] = scaler.fit_transform(
    flaechen[["score_flaeche", "score_infra"]]
)

# Gesamtscore: 40% Fläche, 60% Infrastruktur
flaechen["score_gesamt"] = (
    flaechen["score_flaeche"] * 0.4 +
    flaechen["score_infra"]   * 0.6
)

flaechen = flaechen.sort_values("score_gesamt", ascending=False)

print("\nRanking Top 10:")
print(flaechen[["TNTXT", "area_ha", "dist_autobahn_m", "score_gesamt"]].head(10).round(2))

Nach Infrastruktur-Filter (<2km): 17 Flächen

Ranking Top 10:
                                                   TNTXT  area_ha  \
18226               124592 qm Landwirtschaft (Ackerland)    12.45   
18382                84674 qm Landwirtschaft (Ackerland)     8.47   
14889                79504 qm Landwirtschaft (Ackerland)     7.95   
17881               100631 qm Landwirtschaft (Ackerland)    10.07   
18139                96592 qm Landwirtschaft (Ackerland)     9.66   
18145                88124 qm Landwirtschaft (Ackerland)     8.81   
17627               173466 qm Landwirtschaft (Ackerland)    17.35   
20399  982 qm Gehoelz; 9237 qm Flaeche Gemischter Nut...    10.25   
17213               112624 qm Landwirtschaft (Ackerland)    11.25   
9089   432 qm Industrie Und Gewerbeflaeche (Handel un...     7.13   

       dist_autobahn_m  score_gesamt  
18226          1738.90          0.69  
18382          1514.71          0.65  
14889          1645.09          0.56  
17881          1833.46

In [29]:
# ─────────────────────────────────────────
# FILTERN
# ─────────────────────────────────────────

# Koordinatensystem auf Meter umstellen (UTM Zone 32N für Deutschland)
autobahnen_m   = autobahnen.to_crs(epsg=25832)
bahnlinien_m   = bahnlinien.to_crs(epsg=25832)
landnutzung_m  = landnutzung.to_crs(epsg=25832)

# Fläche in Hektar berechnen
landnutzung_m["area_ha"] = landnutzung_m.geometry.area / 10000

# Filter 1: Nur Flächen größer 7 Hektar
flaechen = landnutzung_m[landnutzung_m["area_ha"] > 7].copy()
print(f"\nNach Größenfilter (>7 ha): {len(flaechen)} Flächen")

# Infrastruktur zu je einer Geometrie zusammenfassen
autobahn_union = unary_union(autobahnen_m.geometry)
bahn_union     = unary_union(bahnlinien_m.geometry)

# Abstände berechnen
flaechen["dist_autobahn_m"] = flaechen.geometry.apply(
    lambda g: g.distance(autobahn_union)
)
flaechen["dist_bahn_m"] = flaechen.geometry.apply(
    lambda g: g.distance(bahn_union)
)

# Filter 2: Nähe zur Infrastruktur (innerhalb 2km von Autobahn oder Bahn)
flaechen = flaechen[
    (flaechen["dist_autobahn_m"] < 2000) |
    (flaechen["dist_bahn_m"] < 2000)
].copy()
print(f"Nach Infrastruktur-Filter (<2km): {len(flaechen)} Flächen")



Nach Größenfilter (>7 ha): 43 Flächen
Nach Infrastruktur-Filter (<2km): 41 Flächen


In [44]:
# Zurück zu WGS84 für die Karte
flaechen_wgs = flaechen.to_crs(epsg=4326)

m = folium.Map(location=KARTE_MITTE, zoom_start=13)

# Autobahnen lila einzeichnen
for _, row in autobahnen.iterrows():
    coords = [(lat, lon) for lon, lat in row.geometry.coords]
    folium.PolyLine(
        coords, color="purple", weight=3,
        opacity=0.7, tooltip="Autobahn"
    ).add_to(m)

# Bahnlinien blau einzeichnen
for _, row in bahnlinien.iterrows():
    coords = [(lat, lon) for lon, lat in row.geometry.coords]
    folium.PolyLine(
        coords, color="blue", weight=2,
        opacity=0.5, tooltip="Bahnlinie"
    ).add_to(m)

# Flächen nach Score einfärben
for _, row in flaechen_wgs.iterrows():
    farbe = farbe_nach_score(row.score_gesamt)
    popup_text = folium.Popup(
        f"<b>{kategorie_nach_score(row.score_gesamt)}</b><br>"
        f"Nutzung: {row.TNTXT[:50]}...<br>"
        f"Fläche: {row.area_ha:.1f} ha<br>"
        f"Abstand Autobahn: {row.dist_autobahn_m:.0f} m<br>"
        f"Abstand Bahn: {row.dist_bahn_m:.0f} m<br>"
        f"Score: {row.score_gesamt:.3f}",
        max_width=250
    )

    # MultiPolygon und Polygon beide unterstützen
    geom = row.geometry
    if geom.geom_type == "MultiPolygon":
        teile = list(geom.geoms)
    else:
        teile = [geom]

    for teil in teile:
        coords = [(lat, lon) for lon, lat in teil.exterior.coords]
        folium.Polygon(
            locations=coords,
            color=farbe,
            fill=True,
            fill_opacity=0.5,
            popup=popup_text
        ).add_to(m)

# Legende
legende_html = """
<div style="
    position: fixed; bottom: 30px; right: 30px; z-index: 1000;
    background-color: white; padding: 15px; border-radius: 8px;
    border: 1px solid #ccc; font-family: Arial, sans-serif;
    font-size: 13px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);">
    <b style="font-size:14px;">🗺️ Legende</b><br><br>
    <span style="display:inline-block; width:30px; height:4px;
        background:purple; margin-right:8px; vertical-align:middle;"></span>
    Autobahn<br><br>
    <span style="display:inline-block; width:30px; height:4px;
        background:blue; margin-right:8px; vertical-align:middle;"></span>
    Bahnlinie<br><br>
    <hr style="margin:8px 0;">
    <b>Flächenbewertung:</b><br><br>
    <span style="display:inline-block; width:16px; height:16px;
        background:green; opacity:0.6; margin-right:8px;
        vertical-align:middle; border:1px solid green;"></span>
    Gut geeignet (Score &gt; 0.66)<br><br>
    <span style="display:inline-block; width:16px; height:16px;
        background:orange; opacity:0.6; margin-right:8px;
        vertical-align:middle; border:1px solid orange;"></span>
    Mittel geeignet (Score 0.33–0.66)<br><br>
    <span style="display:inline-block; width:16px; height:16px;
        background:red; opacity:0.6; margin-right:8px;
        vertical-align:middle; border:1px solid red;"></span>
    Wenig geeignet (Score &lt; 0.33)<br><br>
    <hr style="margin:8px 0;">
    <b>Datenquelle:</b> ALKIS Frankfurt<br>
    <b>Filterkriterien:</b><br>
    • Fläche &gt; 7 ha<br>
    • &lt; 2km von Autobahn/Bahn<br>
    • Nutzung: Acker, Grünland
</div>
"""
m.get_root().html.add_child(folium.Element(legende_html))

m